# Phase 9: Local Interactive Showcase Web App & Diagnostic Simulator
**Course: ES 335 - Machine Learning | Indian Institute of Technology Gandhinagar**
**Authors:** Devarsh Patel (24110245) & Dhrumil Bhutaiya (24110075)

---
## 1. Architectural Overview & Clinical Purpose
Phase 9 transforms our benchmark findings and clinical error taxonomy into a **standalone, zero-dependency medical-grade web application** (`docs/index.html`).

### Key Capabilities:
1. **Diagnostic Studio (Interactive Patch Inspector):** Visualizes 96x96 histology patches with a toggleable red central 32x32 diagnostic bounding box, multi-model probability meters, and feature z-score deviation bars.
2. **Real-Time Clinical Decision Threshold Simulator:** Dynamically computes 2x2 confusion matrices and operational metrics across all **5,000 test patches** at 60 FPS in JavaScript as the threshold slider is dragged.
3. **Forensic Pathology Error Gallery:** Curated visual catalog categorized across our 4 clinical failure tiers with pathologist commentary.
4. **Leaderboard & Generalization Gap Dashboard:** Interactive benchmark comparison, bootstrap 95% confidence intervals, and publication curves.
5. **Zero External Dependencies:** Built with pure HTML5, CSS3, and JavaScript, running immediately in any modern browser and 100% ready for GitHub Pages.

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

PROJECT_ROOT = Path('..').resolve()
DOCS_DIR = PROJECT_ROOT / 'docs'
DATA_DIR = DOCS_DIR / 'data'
PLOTS_DIR = PROJECT_ROOT / 'outputs' / 'plots'

print(f'[OK] Project Root: {PROJECT_ROOT}')
print(f'[OK] Showcase Docs Directory: {DOCS_DIR}')

## 2. Verification of Showcase Data Payloads
We verify that both the compact 5,000-prediction array (`test_predictions.json`) and the 80 curated representative clinical cases (`curated_patches.json`) are valid and structured properly.

In [ ]:
# Load curated patches
with open(DATA_DIR / 'curated_patches.json', 'r', encoding='utf-8') as f:
    curated_patches = json.load(f)

# Load 5,000 test predictions
with open(DATA_DIR / 'test_predictions.json', 'r', encoding='utf-8') as f:
    test_preds = json.load(f)

print(f'[OK] Loaded {len(curated_patches)} curated patches.')
print(f'[OK] Loaded {len(test_preds)} test patch predictions across 5 models.')

df_cur = pd.DataFrame(curated_patches)
print('\nCurated Patches Distribution by Category:')
print(df_cur['category'].value_counts().to_string())

## 3. Decision Threshold Simulation Verification
Here we test the operational logic implemented in `app.js` across the 5,000 test patch predictions in Python. We sweep the decision threshold $\tau \in [0.10, 0.90]$ and measure the clinical trade-off between Sensitivity (Recall) and Precision (PPV).

In [ ]:
thresholds = np.linspace(0.1, 0.9, 9)
sim_results = []

y_true = np.array([p['y'] for p in test_preds])
p_blend = np.array([p['b'] for p in test_preds])

for tau in thresholds:
    y_pred = (p_blend >= tau).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    
    sens = tp / (tp + fn)
    spec = tn / (tn + fp)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    acc = (tp + tn) / len(y_true)
    f1 = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
    
    sim_results.append({
        'Threshold': round(tau, 2),
        'Accuracy': f'{acc*100:.2f}%',
        'Sensitivity': f'{sens*100:.2f}%',
        'Specificity': f'{spec*100:.2f}%',
        'Precision': f'{prec*100:.2f}%',
        'F1-Score': round(f1, 4),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    })

df_sim = pd.DataFrame(sim_results)
print('Clinical Operating Points Sweep (Hybrid Multi-Paradigm Blend):')
print(df_sim.to_string(index=False))

## 4. Visual Inspection Sample of Curated Patches
We display representative patches from each curated group with the central $32 \times 32$ diagnostic bounding box overlaid in red.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
sample_categories = [
    'Benign Normal (True Negative)',
    'Invasive Tumour (True Positive)',
    'False Positive (Benign Flagged)',
    'False Negative (Missed Tumour)'
]

for ax, cat_name in zip(axes, sample_categories):
    sample = next(p for p in curated_patches if p['category'] == cat_name)
    img_path = DOCS_DIR / sample['image_url']
    img = Image.open(img_path)
    
    ax.imshow(img)
    # Add central 32x32 bounding box (from 32 to 64)
    rect = patches.Rectangle((32, 32), 32, 32, linewidth=2, edgecolor='red', facecolor='none', linestyle='--')
    ax.add_patch(rect)
    prob_pct = sample['probabilities']['hybrid_blend'] * 100
    ax.set_title(f"{sample['category']}\nGT: {sample['ground_truth']} | P(Tumor): {prob_pct:.1f}%", fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Sample Curated Patches with Central 32x32 Diagnostic Region', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 5. How to Launch the Local Showcase Web App
To launch the interactive showcase in your browser, execute the local server script:
```bash
python src/showcase_server.py
```
This launches Python's standard HTTP server at `http://127.0.0.1:8080/index.html` and opens your default browser.
To deploy to GitHub Pages later, simply set the repository GitHub Pages source folder to `/docs`.